# LoRA safetensors → Draw Things `.ckpt` (Colab)

Converts a LoRA `.safetensors` file to the Draw Things `.ckpt` format with Draw Things' own converter
(`media-generation-kit-cli lora convert`, pinned revision), built on the free Colab CPU runtime. Use it when you
cannot run the converter locally (it has no native Windows build).

1. **Runtime → Run all** (or run the cells in order). Cell 2 builds the converter: expect **30–60 minutes** the first time.
2. Cell 3 asks you to pick your `.safetensors` file and downloads the `.ckpt`.

**Status: experimental.** The build has not been proven on Colab yet. If a cell fails, copy its last lines into the dtFlow issue.
Provenance: the pinned source revisions are printed by cell 2 and written next to the `.ckpt` as a `.json` file.
Your LoRA is processed inside your own Colab session; nothing is sent anywhere else.

**Runtime:** the free CPU runtime is enough. A T4 GPU runtime also works but gives no speed-up: the conversion and the
Swift build are CPU and disk work, and the converter is built without CUDA. Prefer the CPU runtime so the GPU quota stays free.

In [ ]:
# @title 1. Install Swift toolchain and build libraries
import hashlib
import json
import os
import platform
import shlex
import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

SWIFT_VERSION = "6.0.3"
SWIFT_ROOT = "/opt/swift"


def sh(command, cwd=None, every=1):
    """Run a shell command and stream its output into the cell line by line."""
    print("$", command, flush=True)
    process = subprocess.Popen(
        command,
        shell=True,
        executable="/bin/bash",
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    count = 0
    for line in process.stdout:
        count += 1
        if count % every == 0:
            print(line.rstrip(), flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")


def show_progress(blocks, block_size, total):
    """Print download progress about every 5 percent."""
    if total > 0:
        percent = min(100, blocks * block_size * 100 // total)
        if percent // 5 != show_progress.last:
            show_progress.last = percent // 5
            print(f"  downloaded {percent}%", flush=True)


show_progress.last = -1

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This notebook needs the x86_64 Linux Colab runtime.")

release = {}
with open("/etc/os-release") as handle:
    for line in handle:
        if "=" in line:
            key, value = line.rstrip().split("=", 1)
            release[key] = value.strip('"')
ubuntu = release.get("VERSION_ID", "")
if ubuntu not in ("22.04", "24.04"):
    raise RuntimeError(f"Unsupported Ubuntu {ubuntu!r}; this notebook supports 22.04 and 24.04.")

sh(
    "apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y "
    "git clang libcurl4-openssl-dev libxml2-dev libsqlite3-dev zlib1g-dev libpng-dev libjpeg-dev "
    "libomp-dev libblas-dev liblapack-dev libopenblas-dev libssl-dev pkg-config binutils libc6-dev "
    "libstdc++-12-dev tzdata unzip"
)

folder = f"swift-{SWIFT_VERSION}-RELEASE-ubuntu{ubuntu}"
if not os.path.exists(f"{SWIFT_ROOT}/{folder}/usr/bin/swift"):
    url = (
        f"https://download.swift.org/swift-{SWIFT_VERSION}-release/ubuntu{ubuntu.replace('.', '')}/"
        f"swift-{SWIFT_VERSION}-RELEASE/{folder}.tar.gz"
    )
    os.makedirs(SWIFT_ROOT, exist_ok=True)
    archive = f"/tmp/{folder}.tar.gz"
    print("downloading", url, flush=True)
    urllib.request.urlretrieve(url, archive, show_progress)
    print("extracting (a progress line every 2000 files)", flush=True)
    sh(f"tar -xzvf {archive} -C {SWIFT_ROOT}", every=2000)
    os.remove(archive)

SWIFT_BIN = f"{SWIFT_ROOT}/{folder}/usr/bin"
os.environ["PATH"] = SWIFT_BIN + os.pathsep + os.environ["PATH"]
sh("swift --version")

In [ ]:
# @title 2. Build the converter (30–60 min the first time)
# @markdown Optional: keep the built converter in Google Drive so later sessions skip the build.
use_drive_cache = True  # @param {type:"boolean"}

REPO_URL = "https://github.com/drawthingsai/media-generation-kit.git"
PINNED_REVISION = "37e0b70092ad1a0c1d4c6b24f16e17b73f1b1fb3"
CLI_NAME = "media-generation-kit-cli"
WORK = Path("/content/mgk-build")
SOURCE = WORK / "media-generation-kit"
BINARY = SOURCE / ".build" / "release" / CLI_NAME
DRIVE_CACHE = Path("/content/drive/MyDrive/dtflow-lora-converter")
CACHED_BINARY = DRIVE_CACHE / f"{CLI_NAME}-{PINNED_REVISION[:7]}-swift{SWIFT_VERSION}-ubuntu{ubuntu}"

if use_drive_cache:
    from google.colab import drive

    drive.mount("/content/drive")

if use_drive_cache and CACHED_BINARY.is_file() and not BINARY.is_file():
    BINARY.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CACHED_BINARY, BINARY)
    BINARY.chmod(0o755)
    print("restored the converter from Drive:", CACHED_BINARY)

if not BINARY.is_file():
    WORK.mkdir(parents=True, exist_ok=True)
    if not SOURCE.exists():
        sh(f"git clone --progress {REPO_URL} {SOURCE}")
    sh(f"git -C {SOURCE} checkout {PINNED_REVISION}")
    sh("swift package resolve", cwd=SOURCE)

    # Linux compile fix: ZIPFoundation's in-memory FILE pointer is optional on Linux (fopencookie),
    # but the vendored source passes it straight to fseeko. Unwrap it.
    patched = 0
    for source in SOURCE.glob(
        ".build/checkouts/draw-things-community/Vendors/ZIPFoundation/Sources/ZIPFoundation/Archive+MemoryFile.swift"
    ):
        text = source.read_text()
        before = "      if append {\n        fseeko(result, 0, SEEK_END)"
        after = "      if append, let result = result {\n        fseeko(result, 0, SEEK_END)"
        if after in text:
            patched += 1
        elif text.count(before) == 1:
            source.write_text(text.replace(before, after, 1))
            patched += 1
    if patched != 1:
        raise RuntimeError("ZIPFoundation Linux patch did not apply; upstream changed, the pin needs a new patch.")
    print("ZIPFoundation Linux patch ready")

    jobs = os.cpu_count() or 2
    sh(f"swift build -c release --product {CLI_NAME} -j {jobs}", cwd=SOURCE)
    if use_drive_cache and BINARY.is_file():
        DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BINARY, CACHED_BINARY)
        print("saved the converter to Drive:", CACHED_BINARY)

if not BINARY.is_file():
    raise RuntimeError("The build finished without producing the converter binary.")

draw_things_revision = subprocess.run(
    ["git", "-C", str(SOURCE), "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
print("converter ready:", BINARY)
print("media-generation-kit revision:", PINNED_REVISION)

In [ ]:
# @title 3. Convert a LoRA (pick the file, then download the .ckpt)
# @markdown Scale is applied to the LoRA weights while converting. Leave 1.0 unless you know you need another value.
scale = 1.0  # @param {type:"number"}
# @markdown Leave empty to pick a file from your computer. Or give a path inside Colab / Drive, e.g. /content/drive/MyDrive/my_lora.safetensors
source_path = ""  # @param {type:"string"}
download_result = True  # @param {type:"boolean"}

if source_path.strip():
    source = Path(source_path.strip()).expanduser()
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Pick exactly one .safetensors file.")
    source = Path("/content") / next(iter(uploaded))

if not source.is_file():
    raise RuntimeError(f"Input not found: {source}")
if source.suffix.lower() != ".safetensors":
    print(f"warning: {source.name} does not end in .safetensors")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


out_dir = Path("/content/lora_out")
out_dir.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix=".convert-", dir=out_dir) as staging:
    command = [str(BINARY), "lora", "convert", "--input", str(source), "--output-dir", staging]
    if scale != 1.0:
        command += ["--scale", str(scale)]
    sh(shlex.join(command))
    produced = list(Path(staging).glob("*.ckpt"))
    if len(produced) != 1:
        raise RuntimeError(f"Expected exactly one .ckpt from the converter, found {len(produced)}.")
    target = out_dir / produced[0].name
    if target.exists():
        raise RuntimeError(f"{target} already exists; not overwriting it. Remove it first.")
    produced[0].replace(target)

size = target.stat().st_size
digest = sha256_of(target)
provenance = target.with_suffix(".json")
provenance.write_text(
    json.dumps(
        {
            "output": target.name,
            "size_bytes": size,
            "sha256": digest,
            "input": source.name,
            "input_sha256": sha256_of(source),
            "scale": scale,
            "media_generation_kit_revision": PINNED_REVISION,
            "swift_version": SWIFT_VERSION,
        },
        indent=2,
    )
)
print("output:", target)
print("size:  ", size, "bytes")
print("sha256:", digest)

if download_result:
    from google.colab import files

    files.download(str(target))
    files.download(str(provenance))